In [1]:
# ... existing code ...
import time
from flask import Flask, request, jsonify, Response
import json
from tensor_utils_pybind import  tensor_restore_from_handler_pybind, IPCHandleManager

app = Flask(__name__)

@app.route('/merged_handler', methods=['POST'])
def merged_handler():
    DEBUG = False
    start_time = time.time()

    hidden_states_meta = json.loads(request.form['hidden_states_meta']) 
    if DEBUG:
        print(f"hidden_states_meta: {hidden_states_meta}")
    
    topk_weights_meta = json.loads(request.form['topk_weights_meta']) 
    if DEBUG:
        print(f"topk_weights_meta: {topk_weights_meta}")
    
    topk_ids_meta = json.loads(request.form['topk_ids_meta']) 
    if DEBUG:
        print(f"topk_ids_meta: {topk_ids_meta}")

    handler = request.files['handler'].read()
    handle_manager = IPCHandleManager(handler, hidden_states_meta['device'])


    hidden_states = tensor_restore_from_handler_pybind(handle_manager, hidden_states_meta)
    topk_weights = tensor_restore_from_handler_pybind(handle_manager, topk_weights_meta)
    topk_ids = tensor_restore_from_handler_pybind(handle_manager, topk_ids_meta)
    end_time = time.time()
    
    handle_manager.close_ipc_handle()
    
    print(f"hidden_states: {hidden_states}")
    print(f"topk_weights: {topk_weights}")
    print(f"topk_ids: {topk_ids}")
    print(f"restored 3 tensors in : {(end_time - start_time)*1000} ms")
    
    response = {
        'message':"ok",
    }
    
    return jsonify(response)

@app.route('/merged_single', methods=['POST'])
def test():
    merged_handler = request.files['merged_handler'].read()
    

    hidden_states_meta = json.loads(request.form['hidden_states_meta']) 

    
    print(f"hidden_states_meta: {hidden_states_meta}")

    
    handle_manager = IPCHandleManager(merged_handler, hidden_states_meta['device'])
    
        # 使用同一个handle管理器创建所有tensor
    global hidden_states
    hidden_states = tensor_restore_from_handler_pybind(handle_manager, hidden_states_meta,True)
    
    print(f"merged: {hidden_states}")


    response = {
        'message':"ok",
        # 'restored_tensor':t.cpu().tolist(),  # Convert to list for JSON serialization
    }
    
    return jsonify(response)

if __name__ == '__main__':
    app.run(port=1177)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:1177
Press CTRL+C to quit
Exception ignored in: <function IPCHandleManager.__del__ at 0x72e2c9831d80>
Traceback (most recent call last):
  File "/root/vllm/ipc_handler_demo/exp_field/tensor_utils_pybind.py", line 58, in __del__
    self.close_ipc_handle()
  File "/root/vllm/ipc_handler_demo/exp_field/tensor_utils_pybind.py", line 55, in close_ipc_handle
    ipc_tensor_pybind.close_ipc_handle(self.dev_ptr)
RuntimeError: cudaIpcCloseMemHandle failed: invalid argument
127.0.0.1 - - [07/Jun/2025 12:19:21] "POST /merged_handler HTTP/1.1" 200 -


offset_ptr: 0x72e2a8200000
tensor.data_ptr: 0x72e2a8c00000
offset_ptr: 0x72e2a8a00000
tensor.data_ptr: 0x72e2ac800000
offset_ptr: 0x72e2a8a04000
tensor.data_ptr: 0x72e2ac804000
hidden_states: tensor([[ 0.3281,  0.4315, -1.0429,  ..., -0.5278, -0.1863, -1.2353],
        [-0.7547,  1.2988,  1.0982,  ..., -0.0043, -0.6751,  0.2501],
        [-0.5654, -1.2827,  0.5470,  ...,  0.1394, -1.0574,  0.0928],
        ...,
        [ 0.4220,  0.4585,  0.1369,  ..., -1.1160, -1.6434,  0.0908],
        [ 1.7464, -2.1227, -0.1922,  ..., -0.5449,  0.8911,  1.0948],
        [ 0.2750,  0.2415,  1.6080,  ..., -0.4683, -1.9170,  0.9468]],
       device='cuda:1')
topk_weights: tensor([[56,  9, 36, 29],
        [ 7, 54, 21, 29],
        [16, 21, 48, 16],
        ...,
        [ 6, 50, 55, 37],
        [18,  8,  2, 59],
        [57, 52,  9,  4]], device='cuda:1', dtype=torch.int32)
topk_ids: tensor([[ 0.6680, -1.8906, -0.9492,  0.4180],
        [-1.0625, -1.2891,  0.2656, -0.6250],
        [ 0.8086, -0.5742,  

Exception ignored in: <function IPCHandleManager.__del__ at 0x72e2c9831d80>
Traceback (most recent call last):
  File "/root/vllm/ipc_handler_demo/exp_field/tensor_utils_pybind.py", line 58, in __del__
    self.close_ipc_handle()
  File "/root/vllm/ipc_handler_demo/exp_field/tensor_utils_pybind.py", line 55, in close_ipc_handle
    ipc_tensor_pybind.close_ipc_handle(self.dev_ptr)
RuntimeError: cudaIpcCloseMemHandle failed: invalid argument
127.0.0.1 - - [07/Jun/2025 12:19:21] "POST /merged_handler HTTP/1.1" 200 -


offset_ptr: 0x72e29b200000
tensor.data_ptr: 0x72e2a8c00000
offset_ptr: 0x72e29ba00000
tensor.data_ptr: 0x72e2ac800000
offset_ptr: 0x72e29ba02000
tensor.data_ptr: 0x72e2ac802000
hidden_states: tensor([[ 0.5085, -0.6637,  0.8721,  ..., -0.1993,  1.2322, -0.2183],
        [ 0.2567, -0.1289, -2.6810,  ..., -0.4056, -1.6370, -1.2568],
        [-0.9615,  0.6709, -0.9246,  ..., -0.7930, -0.1826, -0.3114],
        ...,
        [ 1.4385, -0.5238, -1.1098,  ..., -0.8638, -0.4459, -1.0100],
        [ 0.7673,  2.0026,  0.0254,  ...,  0.3831,  0.0618,  0.4125],
        [-0.8063,  1.0232,  0.5609,  ..., -0.7465, -0.3194,  0.5995]],
       device='cuda:1')
topk_weights: tensor([[ 0.9414,  1.3281, -1.6094,  0.3496],
        [-0.7578,  2.7344,  0.0322,  0.1279],
        [ 1.4844,  0.7109,  0.4414,  0.4746],
        ...,
        [ 0.4395,  2.4375, -0.8164,  1.8750],
        [ 0.3340,  0.1934, -0.2637,  1.0938],
        [-0.6406, -1.5469,  0.1689, -0.3613]], device='cuda:1',
       dtype=torch.bfloat16)
